[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lizcodd/fastq2pheno_workshop/blob/main/04_sequence_models/chrombpnet.ipynb)

# What opens chromatin? Sequence models with chromBPNet

**Where we are.** Step 3 compared COLO320DM (*MYC* on ecDNA) with COLO320HSR (*MYC* in a chromosome) and found
thousands of peaks that differ. HOMER found the **AP-1** motif (bound by the JUN/FOS family of TFs) enriched in the
peaks more open in HSR. Enrichment says AP-1 sites and HSR-open peaks go together. It can't say whether the sites
*cause* the opening, or which sites, in which peaks.

**The idea.** Train a model that reads DNA sequence and predicts accessibility, one model per line. The models then
become an experiment you can run in the computer: give both models **the same sequence**, remove or add a motif, and
see how each line's prediction changes. Because the input is identical, a difference between the two answers comes
from what each line's cells respond to, such as which TFs are present and active.

**The tool.** [chromBPNet](https://github.com/kundajelab/chrombpnet) is a deep-learning model that reads DNA
sequence and predicts ATAC-seq signal. Training takes hours of GPU time (Step 4's README shows how), so this notebook
starts from the two trained models and runs them live.

1. What chromBPNet is
2. Did the models learn?
3. What each model relies on: its motifs
4. One peak, base by base
5. Edit the DNA
6. Takeaways

Cells marked **✏️ Try it** have parameters to change.

*In Colab the badge above starts the default runtime (Python 3, CPU), which is all this needs. Run the cells in order.*

## 0. Setup (about 2 minutes)

In [ ]:
# Install logomaker (sequence logos); TensorFlow and the rest come with Colab
import importlib.util, subprocess, sys
if importlib.util.find_spec("logomaker") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "logomaker"], check=True)

In [ ]:
# Download the data (one archive on Google Drive, about 57 MB) and unpack it into data/
import os, tarfile, urllib.request
drive_file_id = "1MRqM3-M6dFEFRrlD5RZYpA9BCI1HjoUj"
if not os.path.exists("data/regions.npz"):
    url = f"https://drive.usercontent.google.com/download?id={drive_file_id}&export=download&confirm=t"
    urllib.request.urlretrieve(url, "chrombpnet_data.tar.gz")
    tarfile.open("chrombpnet_data.tar.gz").extractall(".")
sorted(os.listdir("data"))

In [ ]:
# Load the data and the two models, and define helper functions (no need to read this cell closely)
import json, contextlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
from IPython.display import Image, display
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"; os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

@contextlib.contextmanager
def quiet():                       # TensorFlow prints start-up messages straight to stderr
    fd = os.dup(2); null = os.open(os.devnull, os.O_WRONLY); os.dup2(null, 2)
    try:
        yield
    finally:
        os.dup2(fd, 2); os.close(null); os.close(fd)

with quiet():
    import tensorflow as tf
plt.rcParams.update({"font.size": 14, "axes.spines.top": False, "axes.spines.right": False})
LINES = ["DM", "HSR"]
COLORS = {"DM": "firebrick", "HSR": "steelblue"}

d = np.load("data/regions.npz", allow_pickle=True)      # 750 featured peaks: sequence + per-base tracks
TRACKS = {name: d["tracks"][:, i].astype(np.float32) for i, name in enumerate(d["track_names"])}
featured = pd.DataFrame({k: d[k] for k in ["peak", "chrom", "center", "gene", "call", "log2FC"]})
SEQS = d["seq"]                                          # 2,114 bp around each featured peak's summit
POS = np.arange(-500, 500)                               # tracks cover the central 1,000 bp
peaks = pd.read_csv("data/peaks.tsv.gz", sep="\t")       # every consensus peak: observed and predicted reads
ablation = pd.read_csv("data/ablation.tsv.gz", sep="\t")
motifs = json.load(open("data/motifs.json"))
qc = json.load(open("data/qc.json"))
BACKGROUNDS = [l.strip() for l in open("data/backgrounds.txt") if l.strip()]
with quiet():
    MODELS = {l: tf.keras.models.load_model(f"data/models/chrombpnet_{l}.h5", compile=False) for l in LINES}

def one_hot(seqs):
    lut = np.full(256, 4); lut[[ord(c) for c in "ACGT"]] = np.arange(4)
    eye = np.vstack([np.eye(4, dtype=np.float32), np.zeros((1, 4), np.float32)])
    return eye[lut[np.frombuffer("".join(seqs).encode(), np.uint8)].reshape(len(seqs), -1)]

def predict(seqs):
    # each line's model: predicted log reads (natural log) and the profile (share of cuts at each of 1,000 bp)
    X = one_hot(seqs)
    out = {}
    for l in LINES:
        profile_logits, log_counts = MODELS[l].predict(X, verbose=0)
        p = np.exp(profile_logits - profile_logits.max(axis=1, keepdims=True))
        out[l] = {"log_counts": log_counts[:, 0], "profile": p / p.sum(axis=1, keepdims=True)}
    return out

def plant(motif, seqs=BACKGROUNDS):
    # write the motif into the middle of each background sequence
    a = 1057 - len(motif) // 2
    return [s[:a] + motif + s[a + len(motif):] for s in seqs]

def find_peak(pick):
    # pick: a peak ID or a gene name
    hit = featured[(featured.peak == pick) | (featured.gene == pick)]
    if len(hit) == 0:
        raise ValueError(f"{pick} is not among the featured peaks")
    return hit.index[0]

def contrib_diff(i):
    # HSR minus DM contribution score at each base, summed over 11 bp
    return np.convolve(TRACKS["contrib_HSR"][i].astype(float) - TRACKS["contrib_DM"][i], np.ones(11), "same")

def most_different(i):
    # bp from the summit where the two models' contribution scores differ most
    return int(np.argmax(np.abs(contrib_diff(i)))) - 500

def logo(ax, values, seq, title=None):
    df = pd.DataFrame(0.0, index=range(len(seq)), columns=list("ACGT"))
    for i, b in enumerate(seq):
        if b in "ACGT":
            df.loc[i, b] = values[i]
    logomaker.Logo(df, ax=ax, color_scheme="classic")
    if title:
        ax.set_title(title, fontsize=12, loc="left")
    ax.set_xticks([])

smooth = lambda v, n=10: np.convolve(v, np.ones(n) / n, mode="same")
print(f"{len(featured)} featured peaks, {len(peaks):,} peaks in all")

## 1. What chromBPNet is

chromBPNet reads **2,114 bp of DNA sequence** and predicts two things about the central 1,000 bp:

- **counts**: how many reads the region gets (how accessible it is);
- **profile**: where exactly Tn5 cuts, base by base.

**Under the hood** it is a **convolutional neural network** (diagram below). Layer 1 slides 512 learned filters,
each 21 bp wide, along the sequence, much like scanning it with motifs. Layers 2–9 combine those hits over growing
distances, so the model can learn which motifs occur together and how far apart. Two heads turn the result into the
profile and the counts.

**The bias model.** Tn5 has its own sequence preference, which the model would otherwise learn as if it were biology.
A pretrained **bias model** (here ENCODE's, from K562 ATAC-seq) captures it: the same design, but too small to see
more than about 150 bp. During training the two models' predictions are added together and only the main model
learns, so it has to explain everything except Tn5's preference. After training the bias model is dropped, and the
main model alone (**bias-free**) is used in this notebook.

**Why 2,114 bp?** Each layer loses a little at both edges, where its filter would run off the sequence. Across all
layers that adds up to 1,114 bp, leaving the central 1,000 bp, where every base is predicted from its full
surrounding sequence.

**Our models.** One per line, each trained on that line's reads (both replicates pooled), on the same set of
regions: the consensus peaks plus GC-matched background regions. Amplicons are left out, since the model can't know
a region has 100 copies. Training used most chromosomes; **chr1, chr3 and chr6 were held out** and never seen, so
predictions there are a fair test.

In [ ]:
display(Image("data/diagram.png"))

## 2. Did the models learn?

A model that learned nothing still makes predictions, so check before trusting it.

**Training.** The **loss** measures how far the predictions are from the data (lower is better). Training loss is
measured on the chromosomes the model learns from, validation loss on two chromosomes it never trains on (chr8 and
chr20). Training loss keeps falling as the model memorizes its training data; validation loss is the one that counts.
Each pass through the training data is an **epoch**. chromBPNet stops after 5 epochs without a new best validation
loss (its default **patience**) and keeps the model from the best epoch (dashed line).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, l in zip(axes, LINES):
    t = qc[l]["training"]
    epoch = np.array(t["epoch"]) + 1
    best = epoch[int(np.argmin(t["val_loss"]))]
    ax.plot(epoch, t["loss"], color="gray", label="training chromosomes")
    ax.plot(epoch, t["val_loss"], color=COLORS[l], label="validation chromosomes")
    ax.axvline(best, color="black", ls="--", lw=1)
    ax.set_title(f"{l} model: best epoch {best}, stopped at {epoch[-1]}", fontsize=13); ax.set_xlabel("epoch")
axes[0].set_ylabel("loss"); axes[0].legend(frameon=False, fontsize=11)
plt.tight_layout(); plt.show()

**Accuracy.** On the test chromosomes, predicted vs observed reads for every peak. A Pearson r of 0.6–0.8 is
typical for bulk ATAC-seq.

In [ ]:
test = peaks[peaks.chromosomes == "test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, l in zip(axes, LINES):
    obs = np.log(test[f"reads_{l}"] + 1)
    r = np.corrcoef(obs, test[f"pred_{l}"])[0, 1]
    ax.hexbin(obs, test[f"pred_{l}"], gridsize=60, bins="log", cmap="coolwarm", mincnt=1)
    ax.set_xlabel("observed log reads"); ax.set_ylabel("predicted log reads")
    ax.set_title(f"{l} model: {len(test):,} test peaks, r = {r:.2f}", fontsize=13)
plt.tight_layout(); plt.show()

**Bias removed?** Plant Tn5's preferred sequence, or a CTCF site, into the middle of 16 background sequences
(GC-matched regions with no peak) and average each bias-free model's predicted cut profile around it. A bound TF
protects its site from Tn5, so a real site should leave a dip in the cuts (a **footprint**). If the bias was removed,
Tn5's own preferred sequence should leave no pattern.

In [ ]:
PLANTS = {"Tn5's preferred sequence": "GTGCACAGTTCTAGAGTGTGCAG", "CTCF site": "CCACTAGGGGGCGC"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (name, m) in zip(axes, PLANTS.items()):
    pr = predict(plant(m))
    for l in LINES:
        ax.plot(np.arange(-100, 100), pr[l]["profile"].mean(axis=0)[400:600], color=COLORS[l], label=f"{l} model")
    ax.set_title(f"planted: {name}", fontsize=13); ax.set_xlabel("bp from the planted sequence")
axes[0].set_ylabel("predicted share of cuts"); axes[0].legend(frameon=False, fontsize=11)
plt.tight_layout(); plt.show()

## 3. What each model relies on: its motifs

For every peak, chromBPNet can score how much **each base** contributes to the prediction (its **contribution
score**). **TF-MoDISco** collects the high-scoring stretches across all peaks, clusters similar ones into motifs and
ranks them by how many sites they have. Only then does it look up the closest known motif in a database.

**Read the logos, not the names.** The name is the database's best guess, and a poor match (high q-value) means
little. Several motifs can also be variants of one. In the HSR panel, #1, #3, #10 and #12 contain `TTTAT`, and #4, #6
and #9 contain `ATAAA`, the same motif read from the other strand, under names from CDX1 and CDX2 to CEBPA and SOX15.
It is the motif of **CDX2**, a master regulator of intestinal identity (COLO320 is a colorectal cancer line), and it
tops the DM list too. Both models rely on mostly the same motifs. **AP-1** (`TGAGTCA`, #16) is the exception: DM's
list has no AP-1 motif (its ATF1 matches are a different motif, `TGACGTCA`).

**✏️ Try it.** Switch `line` to `"DM"` and compare, or change `top_n`.

In [ ]:
line, top_n = "HSR", 16

pats = motifs[line][:top_n]
ncol = 4; nrow = max(1, int(np.ceil(len(pats) / ncol)))
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 2.4 * nrow), squeeze=False)
for k, (ax, m) in enumerate(zip(axes.flat, pats), start=1):
    logomaker.Logo(pd.DataFrame(m["cwm"], columns=list("ACGT")), ax=ax, color_scheme="classic")
    best = f"{m['matches'][0][0].split('_')[0]} (q = {m['matches'][0][1]:.1g})" if m["matches"] else "no match"
    ax.set_title(f"#{k}: {m['n_seqlets']:,} sites\n{best}", fontsize=10, loc="left")
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[len(pats):]:
    ax.axis("off")
plt.suptitle(f"{line} model", y=1.0); plt.tight_layout(); plt.show()

## 4. One peak, base by base

First, let the models point to good examples. For each featured peak on a test chromosome (unseen during training),
the next cell compares the difference the two models **predict** (log2 DM / HSR, corrected for sequencing depth) with
the one Step 3 **observed**, and finds the 21 bp **stretch** where the two models' contribution scores differ most.
Each table keeps the peaks where the models agree with Step 3 and shows the five with the most strongly differing
stretch.

In [ ]:
# chromBPNet predicts each line's raw read counts, so DM's deeper sequencing adds about the same amount to every
# predicted DM/HSR difference. As with DESeq2's size factors, remove it: the median predicted difference over all peaks.
depth_offset = np.median(peaks.pred_DM - peaks.pred_HSR) / np.log(2)

t = featured[featured.chrom.isin(["chr1", "chr3", "chr6"])].copy()       # test chromosomes only
pr = predict(list(SEQS[t.index]))
t["predicted log2FC"] = (pr["DM"]["log_counts"] - pr["HSR"]["log_counts"]) / np.log(2) - depth_offset
t["position"] = [most_different(i) for i in t.index]
t["stretch"] = [SEQS[i][1057 + c - 10:1057 + c + 11] for i, c in zip(t.index, t.position)]
diff = np.array([contrib_diff(i)[c + 500] for i, c in zip(t.index, t.position)])
t["weighted more by"] = np.where(diff > 0, "HSR", "DM")
t["strength"] = np.abs(diff)

agree = {"up in HSR": t[(t.call == "up in HSR") & (t["predicted log2FC"] <= -1) & (t["weighted more by"] == "HSR")],
         "up in DM": t[(t.call == "up in DM") & (t["predicted log2FC"] >= 1) & (t["weighted more by"] == "DM")],
         "not significant": t[(t.call == "not significant") & (t["predicted log2FC"].abs() <= 0.5)]}
cols = ["peak", "gene", "log2FC", "predicted log2FC", "position", "stretch", "weighted more by", "strength"]
for call, q in agree.items():
    print(f"{call}: {len(q)} peaks agree; the five with the strongest differing stretch")
    display(q.sort_values("strength", ascending=False).head(5)[cols].rename(columns={"log2FC": "observed log2FC"}).round(2))

Now look at one peak, the first up-in-HSR row. From top to bottom: observed Tn5 cuts in each line, then each
model's contribution scores, then a zoom (gold) where each base is drawn as its letter, scaled by its contribution.

**✏️ Try it.** Set `pick` to any peak in the tables, for example `"CMC1"` (up in DM, a CTCF site the DM model relies
on) or `"Interval_2078"` (another AP-1 peak up in HSR). `zoom_center` (bp from the summit) moves the zoom; `None`
centers it where the two models differ most.

In [ ]:
pick = "Interval_87909"
zoom_center, zoom_width = None, 50

i = find_peak(pick)
p = featured.loc[i]
if zoom_center is None:
    zoom_center = most_different(i)
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
for l in LINES:
    axes[0].plot(POS, smooth(TRACKS[f"cuts_{l}"][i].astype(float), 20), color=COLORS[l], label=l)
    axes[1 if l == "DM" else 2].plot(POS, TRACKS[f"contrib_{l}"][i], color=COLORS[l], lw=0.8)
axes[0].set_ylabel("observed\ncuts per bp"); axes[0].legend(frameon=False)
axes[1].set_ylabel("DM model\ncontribution"); axes[2].set_ylabel("HSR model\ncontribution")
lim = max(np.abs(TRACKS[f"contrib_{l}"][i]).max() for l in LINES) * 1.1
axes[1].set_ylim(-lim, lim); axes[2].set_ylim(-lim, lim); axes[2].set_xlabel("bp from peak summit")
axes[0].set_title(f"{p.peak}, {p.gene} ({p.chrom}:{p.center:,}): {p.call}", loc="left", fontsize=13)
for ax in axes:
    ax.axvspan(zoom_center - zoom_width // 2, zoom_center + zoom_width // 2, color="gold", alpha=0.3)
plt.show()

lo = 500 + zoom_center - zoom_width // 2
seq = SEQS[i][557 + lo:557 + lo + zoom_width]            # the 2,114 bp sequence starts 557 bp before the tracks
fig, axes = plt.subplots(2, 1, figsize=(14, 4))
lim = max(np.abs(TRACKS[f"contrib_{l}"][i][lo:lo + zoom_width]).max() for l in LINES) * 1.1
for ax, l in zip(axes, LINES):
    logo(ax, TRACKS[f"contrib_{l}"][i][lo:lo + zoom_width], seq, f"{l} model")
    ax.set_ylim(-lim / 4, lim)
plt.tight_layout(); plt.show()
print("zoomed sequence:", seq)

The tall letters in the HSR model spell **TGACTCA**, an AP-1 site. The DM model gives the same bases almost no
weight.

## 5. Edit the DNA

**Scramble the site.** Shuffle the AP-1 site's letters and predict the peak again with both models. If the site is
why the peak is open in HSR, the HSR model's prediction should drop and the DM model's barely change. (Predicted cuts
look smoother than the observed ones: the bias-free model leaves out Tn5's own preference.)

**✏️ Try it.** For `"Interval_2078"`, scramble `"TGAGTCA"`. Any `site` works if it occurs in the peak.

In [ ]:
pick = "Interval_87909"
site = "TGACTCA"

i = find_peak(pick)
before = SEQS[i]
at = before.find(site, 1057 - 400, 1057 + 400)
if at < 0:
    raise ValueError(f"{site} is not within 400 bp of this peak's center")
rng = np.random.default_rng(1)
after = before[:at] + "".join(rng.permutation(list(site))) + before[at + len(site):]
print("scrambled", before[at:at + len(site)], "->", after[at:at + len(site)], f"at {at - 1057:+d} bp")
pr = predict([before, after])
display(pd.DataFrame({f"{l} model": {"predicted reads, change (%)": 100 * (np.exp(pr[l]["log_counts"][1] - pr[l]["log_counts"][0]) - 1)}
                      for l in LINES}).round(1))
fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True, sharey=True)
for ax, l in zip(axes, LINES):
    for k, (label, ls) in enumerate([("original", "-"), ("scrambled", "--")]):
        ax.plot(POS, smooth(pr[l]["profile"][k] * np.exp(pr[l]["log_counts"][k])), ls, color=COLORS[l], label=label)
    ax.axvline(at - 1057, color="gray", lw=0.8); ax.set_ylabel(f"{l} model\npredicted cuts"); ax.legend(frameon=False, fontsize=11)
axes[1].set_xlabel("bp from peak summit"); plt.tight_layout(); plt.show()

**Plant a motif.** Now the other way round: write a motif into the 16 background sequences and see how much each
model's predicted reads rise. The same letters scrambled are the control.

**✏️ Try it.** Compare AP-1 with motifs both models rely on: CDX `TTTATGGC`, CTCF `CCACTAGGGGGCGC`. Or two AP-1
sites, `ATGAGTCATGGATGAGTCAT`.

In [ ]:
motif = "ATGAGTCAT"

scrambled = "".join(np.random.default_rng(2).permutation(list(motif)))
pr0, pr1, pr2 = predict(BACKGROUNDS), predict(plant(motif)), predict(plant(scrambled))
display(pd.DataFrame({f"{l} model": {f"planted {motif}: predicted reads, change (%)": 100 * (np.exp(np.mean(pr1[l]["log_counts"] - pr0[l]["log_counts"])) - 1),
                                     f"scrambled {scrambled}: change (%)": 100 * (np.exp(np.mean(pr2[l]["log_counts"] - pr0[l]["log_counts"])) - 1)}
                      for l in LINES}).round(1))

**Across all peaks.** One peak could be a lucky pick. The same scramble was run on every peak with an AP-1 site
near its center, about 4,800 peaks (on Engaging, `prep/ablation.py`), with CTCF sites for comparison. The control
scrambles a random 6 bp stretch of the same peaks; each bar is the motif's effect minus the control's, with a 95%
interval from resampling peaks. Below zero means the site keeps the peak open.

In [ ]:
rng = np.random.default_rng(0)
rows = []
for name, motif_name in [("AP-1", "AP-1 (TGASTCA)"), ("CTCF", "CTCF core (CCASYAGRKGG)")]:
    g = ablation[ablation.motif == motif_name]
    m, c = g[g.kind == "motif"], g[g.kind == "control"]
    for l in LINES:
        eff = m[f"change_{l}"].values - c[f"change_{l}"].values      # motif and control rows are the same peaks, in order
        boots = [eff[rng.integers(0, len(eff), len(eff))].mean() for _ in range(1000)]
        rows.append({"motif": name, "model": l, "peaks": len(m), "effect": eff.mean(),
                     "low": np.percentile(boots, 2.5), "high": np.percentile(boots, 97.5)})
res = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8, 3.5))
for k, l in enumerate(LINES):
    r = res[res.model == l]
    y = np.arange(len(r)) + (k - 0.5) * 0.4
    ax.barh(y, r.effect, 0.4, color=COLORS[l], label=f"{l} model",
            xerr=[r.effect - r.low, r.high - r.effect], error_kw={"lw": 1})
ax.set_yticks(np.arange(2), ["AP-1", "CTCF"]); ax.invert_yaxis(); ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("change in predicted reads when\nthe site is scrambled (log2)"); ax.legend(frameon=False, fontsize=11)
plt.tight_layout(); plt.show()
res.round(3)

## 6. Takeaways

- **Mostly shared rules.** Both lines' models rely on the same main motifs: CDX2 (intestinal identity), CTCF and
  others. **AP-1 is the clear difference**, used by HSR and nearly ignored by DM.
- **From enrichment to cause, in the model.** HOMER said AP-1 sites go with HSR-open peaks. The chromBPNet models find these same motifs de novo, show which
  bases matter in a given peak, and that removing an AP-1 site lowers the HSR prediction, in one peak and across
  thousands.
- **A model is a hypothesis, not a measurement.** It says which sequence each line's cells respond to, not which TF
  binds: AP-1 sites are bound by many JUN, FOS and ATF family members. Each line has one model, trained once on one
  split of the chromosomes.